# Lecture 13: Path constraints: order, penalties and events

**The challenge.** A robot joint has end stops. A cart runs on a track of
finite length. A spacecraft must not hit the planet it swings past, an
elevator must not exceed its rated speed, a rocket must keep the dynamic
pressure on its structure below a limit. These are limits on the *state*,
not the control: $g(t, x) \le 0$ at every instant. Nothing in Lectures 2 to
4 handles them. The minimum principle constrains the control at each moment,
but a state limit can only be respected by acting *in advance*: braking
before the wall, not at it. A state constraint couples the control now to the
state later, and that changes the structure of the solution.

This lecture treats state limits the robust, approximate way: measure how
"hard" a limit is (its **order**), replace it by a growing penalty, and make
sure the integrator never misses a crossing. Lecture 14 then solves them
exactly.

**After this lecture you can**

- declare path constraints and read their order;
- solve a constrained problem by exterior-penalty continuation, and explain
  why the penalty solution approaches the true one from below;
- use the constraint events, `terminate`, and the violation reports;
- explain why crossings inside a step are hard to detect, and how the
  package detects them.

## 1. A wall, and a mathematician who left Göttingen

The classic test problem is due to Arthur Bryson, Walter Denham and Stuart
Dreyfus, in a 1963 paper that started the systematic study of state
constraints in optimal control. Bryson, at Harvard and later Stanford,
co-wrote the textbook *Applied Optimal Control* with Yu-Chi Ho and gave his
name to "Bryson's rule" for choosing LQR weights. Dreyfus worked at RAND with
Bellman, and later, with his philosopher brother Hubert, wrote *Mind over
Machine* (1986), a sceptical book about artificial intelligence. Their
problem: a mass with $\ddot x = u$ moves from $x = 0$ with velocity $1$ back
to $x = 0$ with velocity $-1$ in unit time, minimizing $\int u^2/2$, but must
stay behind a wall at $x = \ell$.

The approximate remedy is older. In 1943 Richard Courant, addressing the
American Mathematical Society, proposed handling a constraint by adding a
large multiple of its violation to the functional being minimized: a
**penalty**. The same lecture sketched what became the finite element method.
Courant had built the mathematics institute at Göttingen into one of the
great centres of the field. Because he was Jewish, the Nazi government
suspended him in 1933, and he emigrated to New York. There he founded what is
now the Courant Institute at NYU.

## 2. The order of a constraint

Differentiate $g$ along the dynamics until a control appears. The number of
derivatives needed is the constraint's **order**. A velocity limit on
$\ddot x = u$ is order 1 ($\dot g = u$); a position limit is order 2
($\ddot g = u$); a position limit on a jerk-controlled axis,
$\dddot x = u$, is order 3. The order measures how far in advance the
control must act: a high-order limit can only be respected by steering long
before it is reached. Declare constraints with `PathConstraint(expr, name,
penalty)`, meaning `expr <= 0`; `derive` finds the order:

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control import examples
from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import OptimalControlProblem, PathConstraint
from auto_optimal_control.shooting import SingleShooting, parameter_continuation

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t, x, v, a, u = sp.symbols("t x v a u", real=True)


def axis(states, dynamics, limit, name):
    return OptimalControlProblem(
        time=t, states=states, controls=[u], dynamics=dynamics, running_cost=u**2 / 2,
        initial_state=[0] * len(states), final_state={states[0]: 1, **{s: 0 for s in states[1:]}},
        final_time=1, path_constraints=[PathConstraint(limit, name)])


cases = [
    ("speed limit, x'' = u", axis([x, v], [v, u], v - 1, "v_max")),
    ("end stop, x'' = u", axis([x, v], [v, u], x - 1.2, "x_max")),
    ("end stop, x''' = u", axis([x, v, a], [v, a, u], x - 1.2, "x_max")),
]
for label, problem in cases:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        c = derive(problem).path_constraints[0]
    print(f"{label:22s} order {c.order}: derivatives {c.derivatives}")
    for w in caught:
        print("    warning:", str(w.message)[:110], "...")

Order 3 earns a warning, for a reason Lecture 14 explains: optimal solutions
that ride such a limit generally *chatter*. A limit no control can affect at
all is an error, caught at derivation:

In [ ]:
y = sp.Symbol("y", real=True)
try:
    derive(OptimalControlProblem(
        time=t, states=[x, v, y], controls=[u], dynamics=[v, u, 0], running_cost=u**2 / 2,
        initial_state=[0, 0, 0], final_time=1, path_constraints=[PathConstraint(y - 1, "y")]))
except ValueError as err:
    print("ValueError:", err)

## 3. The exterior penalty

Give the constraint a weight $\rho$ (a runtime parameter). The package then
adds $\tfrac{\rho}{2}\max(0, g)^2$ to the running cost: nothing while the
limit holds, a growing price for violating it. As $\rho \to \infty$ the
penalized optimum approaches the constrained one. It approaches *from below*,
because each penalized solution is allowed to violate the limit slightly, and
that is always at least as cheap.

The penalty switches on and off as $g$ crosses zero, so each constraint gets
its own **mode flag**, like a bang-bang control (Lecture 7). Each crossing is
an event, located exactly. The dynamics are continuous there (the penalty
and its gradient vanish at $g = 0$), so the STM does not jump.

The package's `bryson_denham` example carries a penalty weight `rho`. The
closed form is known: for $\ell \ge 1/4$ the wall is never reached; for
$1/6 \le \ell < 1/4$ the mass *touches* it once, at $t = 1/2$; for
$\ell < 1/6$ it *rides* it on the interval $[3\ell, 1 - 3\ell]$, with cost
$4/(9\ell)$.

In [ ]:
bd = derive(examples.bryson_denham())
k = build_kernels(bd)
print("constraint:", bd.path_constraints[0].name, " order", bd.path_constraints[0].order,
      " penalty weight:", bd.path_constraints[0].penalty)
print("mode flags:", k.N_MODES, "(", k.N_SWITCHING, "switching +", k.N_CONSTRAINTS, "constraint )")

ell = 0.1
free = SingleShooting(k, [ell, 0.0]).solve([0.0, 2.0, 0.0, 0.0])
print(f"rho = 0: J = {k.propagate(0, free.z0, 1.0, [ell, 0.0], cost=True)['cost']:.6f}, "
      f"max violation {free.max_violation:.3f} (the unconstrained answer, through the wall)")
path = parameter_continuation(k, [ell, 1.0], "rho", np.logspace(0, 7, 29), free.v,
                              solve_options=dict(tol=1e-8))
rhos = np.array([r for r, _ in path])
costs = np.array([k.propagate(0, s.z0, 1.0, [ell, r], cost=True)["cost"] for r, s in path])
viols = np.array([s.max_violation for _, s in path])
print(f"rho = {rhos[-1]:.0e}: J = {costs[-1]:.6f}, exact 4/(9 l) = {4 / (9 * ell):.6f}, "
      f"violation {viols[-1]:.1e}")

In [ ]:
fig, (left, mid, right) = plt.subplots(1, 3, figsize=(12, 3.3))
for (r, s), color in zip([path[4], path[12], path[-1]], [GRAY, ORANGE, BLUE]):
    tr = SingleShooting(k, [ell, r]).trajectory(s.v, n_points=400)
    left.plot(tr["t"], tr["z"][:, 0], color=color, lw=2, label=f"rho = {r:.0e}")
left.axhline(ell, color="k", lw=1)
left.set(xlabel="t", ylabel="x")
left.legend(frameon=False, fontsize=8)
mid.loglog(rhos, 4 / (9 * ell) - costs, "o-", color=BLUE, ms=3)
mid.set(xlabel="rho", ylabel="exact J - penalty J")
right.loglog(rhos, viols, "o-", color=ORANGE, ms=3)
right.set(xlabel="rho", ylabel="max violation")
for a_ in (left, mid, right):
    a_.grid(alpha=0.3, which="both")
fig.tight_layout()

The penalty solution flattens against the wall, its cost climbs towards
$4/(9\ell)$ from below, and the violation falls, both roughly as a power of
$\rho$. The price of a penalty is conditioning. Near the limit the penalized
dynamics stiffen, and single shooting's sensitivity grows like
$e^{\sqrt{\rho}\,T}$. Continuation in $\rho$ stalls somewhere: here it reaches
$10^7$, giving cost accurate to about $10^{-3}$ relative. Exactness takes a
different method, the subject of Lecture 14. But note what the penalty
solution *does* reveal: *where* the limit is active. That is the structure
Lecture 14 needs.

## 4. Never missing a crossing

An integrator checks a guard's sign at the end of each step. Far from a
limit, steps are long, and a brief violation can begin and end inside one
step, unnoticed. The penalty would then be off where it should be on, the
residual run and the STM run would disagree about the dynamics, and Newton
would stall. Two mechanisms prevent this:

- Every constraint also gets an event on $\dot g$, the peaks and troughs of
  $g$. FNI splits a step at every root it finds, so between consecutive
  extrema $g$ is monotone and crosses zero at most once, and that crossing
  is found.
- A peak and a trough can still fall in the same step. For that case the
  integrator also checks the sign of $g$ at `n_interior` points inside each
  step (default 4, about three extra evaluations per step with DOP853).

`propagate` reports every crossing, and `trajectory` samples $g$:

In [ ]:
r_last, s_last = path[-1]
pr = k.propagate(0.0, s_last.z0, 1.0, [ell, r_last])
print("crossings of g = 0 at t =", np.round(pr["constraint_times"], 6),
      "(constraint", pr["constraint_index"], ")")
print("exact boundary arc: [3l, 1 - 3l] =", [3 * ell, 1 - 3 * ell])

Four crossings, not two. The penalized solution leans slightly past the
wall near the ends of the arc and dips back just inside it in the middle,
where the penalty has pushed it back hardest. The first and last crossings
bracket the exact boundary arc $[0.3, 0.7]$, approaching it as $\rho$ grows.
Reading a structure from a penalty solution means looking past such dips,
which Lecture 14 does.

## 5. Termination: rejecting violators outright

Sometimes a constraint should not be priced but forbidden. A trajectory that
passes through the planet is not a slightly worse trajectory; it is
nonsense. `terminate=True` (or a list of constraint indices or names) stops
propagation at the first violation, with status `-4` and the time of the
violation, and `SingleShooting(terminate=...)` then rejects the guess. For a
gradient-free or random search this is the natural "death penalty": the
candidate is simply discarded. It works for a constraint of any order. It
cannot, however, reach an optimum that *rides* the limit, because no
unconstrained extremal does; for that, Lecture 14.

In [ ]:
viol = k.propagate(0.0, free.z0, 1.0, [ell, 0.0], terminate=True)
print(f"unconstrained extremal with terminate: status {viol['status']} at t = {viol['t']:.4f}")
print("STATUS_VIOLATED =", k.STATUS_VIOLATED)
shoot_t = SingleShooting(k, [ell, 0.0], terminate=["x_max"])
try:
    shoot_t.residual(free.v)
except Exception as err:
    print(type(err).__name__ + ":", err)

Without a penalty weight at all (`PathConstraint(g, name)` with no
`penalty`), a constraint only **monitors**: crossings are reported and
`max_violation` is computed, but the extremal is unchanged.
`SingleShooting.constraint_violation(v)` gives the largest violation per
constraint, sampled densely and between every pair of crossings.

## Summary

- Path constraints $g(t, x) \le 0$ are declared with `PathConstraint`; their
  order is the number of derivatives before a control appears (`derive`
  reports it, warns at order 3, and rejects unreachable limits).
- The exterior penalty $\tfrac{\rho}{2}\max(0, g)^2$, raised by continuation,
  approaches the constrained optimum from below, at a cost in conditioning
  that grows with $\rho$; it reveals where the limit is active.
- Crossings are events with their own mode flags; extrema of $g$ split
  steps, and interior sign checks catch the rest.
- `terminate` rejects violating trajectories (status `-4`), the natural
  constraint handling for random and gradient-free search.

## Exercises

1. Repeat the penalty continuation for $\ell = 0.2$ (the touch regime). How
   does the violation behave as $\rho$ grows, compared with $\ell = 0.1$?
2. Plot $g(t)$ along the penalty solution at $\rho = 10^4$ with
   `trajectory`, and mark the crossings. Where is the violation largest?
3. Pose the jerk-controlled end stop of section 2 with a penalty and
   continue in $\rho$. Plot the control near the limit as $\rho$ grows. What
   do you see? (Lecture 14 explains it.)

## Further reading

- A. E. Bryson, W. F. Denham and S. E. Dreyfus, "Optimal programming
  problems with inequality constraints I: necessary conditions for extremal
  solutions", *AIAA Journal* 1(11), 1963.
- R. Courant, "Variational methods for the solution of problems of
  equilibrium and vibrations", *Bulletin of the AMS* 49, 1943.
- C. Reid, *Courant in Göttingen and New York* (Springer, 1976).
- A. E. Bryson and Y.-C. Ho, *Applied Optimal Control*, section 3.11.